# TPC Ablation Study - Google Colab

**Steps:**
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells 1-4 to setup
3. **IMPORTANT:** Run cell 5 to find your data location
4. Update MIMIC_ROOT in cell 6 if data is elsewhere
5. Run cell 7 to verify data files
6. Run cell 8 to start training (2-4 hours)

In [ ]:
# Clone repository
!git clone https://github.com/tarakjc2c/PyHealth.git
%cd PyHealth
!git checkout pr-1028

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install litdata polars pandas dask mne rdkit peft transformers ogb -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# FIND YOUR MIMIC-IV DATA LOCATION
import os
from pathlib import Path

print('Searching for MIMIC-IV data in Google Drive...')
print('This may take a minute.\n')

# Search common locations
base_path = Path('/content/drive/MyDrive')
possible_locations = [
    base_path / 'mimic-iv',
    base_path / 'MIMIC-IV',
    base_path / 'mimic_iv',
    base_path / 'data' / 'mimic-iv',
    base_path / 'datasets' / 'mimic-iv',
]

found_path = None
for path in possible_locations:
    if path.exists() and (path / 'hosp').exists() and (path / 'icu').exists():
        found_path = str(path)
        print(f'✓ FOUND: {found_path}')
        hosp_files = len(list((path / 'hosp').glob('*.csv.gz')))
        icu_files = len(list((path / 'icu').glob('*.csv.gz')))
        print(f'  hosp/: {hosp_files} files')
        print(f'  icu/: {icu_files} files')
        break

if found_path:
    print(f'\n✓ Use this path in the next cell:')
    print(f'MIMIC_ROOT = "{found_path}"')
else:
    print('✗ MIMIC-IV data not found in common locations')
    print('\nPlease manually search your Drive:')
    print('!find /content/drive/MyDrive -name "chartevents.csv.gz" 2>/dev/null | head -5')
    print('\nOr browse your Drive to find where you uploaded the mimic-iv folder')

In [ ]:
# UPDATE THIS PATH based on cell above!
MIMIC_ROOT = '/content/drive/MyDrive/mimic-iv'  # <-- CHANGE IF NEEDED

# Fix paths for Colab
script = 'examples/length_of_stay/length_of_stay_mimic4_tpc.py'

with open(script, 'r') as f:
    lines = f.readlines()

# Replace path definitions
new_lines = []
for line in lines:
    if 'MIMIC_ROOT = r"C:' in line:
        new_lines.append(f'MIMIC_ROOT = "{MIMIC_ROOT}"\n')
    elif 'CACHE_PATH = r"C:' in line:
        new_lines.append('CACHE_PATH = "/content/tpc_cache"\n')
    elif 'OUTPUT_DIR = "tpc_ablation_results"' in line:
        new_lines.append('OUTPUT_DIR = "/content/drive/MyDrive/tpc_ablation_results"\n')
    else:
        new_lines.append(line)

with open(script, 'w') as f:
    f.writelines(new_lines)

print(f'✓ Script configured with MIMIC_ROOT = {MIMIC_ROOT}')

In [ ]:
# Verify data exists
import os

critical_files = [
    'hosp/diagnoses_icd.csv.gz',
    'icu/chartevents.csv.gz',
    'icu/icustays.csv.gz'
]

all_found = True
for f in critical_files:
    path = os.path.join(MIMIC_ROOT, f)
    if os.path.exists(path):
        print(f'✓ {f}')
    else:
        print(f'✗ MISSING: {f}')
        all_found = False

if all_found:
    print('\n✓ All critical files found! Ready to run.')
else:
    print('\n✗ Some files missing. Check MIMIC_ROOT path above.')

In [ ]:
# Run ablation study (2-4 hours)
!python examples/length_of_stay/length_of_stay_mimic4_tpc.py

## Download Results

Results are in MyDrive/tpc_ablation_results/
Download them with the cell below:

In [ ]:
from google.colab import files
import os

result_dir = '/content/drive/MyDrive/tpc_ablation_results'
for filename in ['ablation_results.json', 'mc_dropout_results.json']:
    filepath = os.path.join(result_dir, filename)
    if os.path.exists(filepath):
        print(f'Downloading: {filename}')
        files.download(filepath)